In [1]:
import cv2
import time
import numpy as np
from PIL import Image
from ultralytics import YOLO
import easyocr
from transformers import BlipProcessor, BlipForConditionalGeneration
import pyttsx3

In [2]:
# Focale calibrée de la caméra (voir cellule de calibration plus bas pour la recalculer)
FOCAL_LENGTH = 700

# Hauteur réelle moyenne des objets (en mètres)
REAL_HEIGHTS = {
    "person": 1.70,
    "bicycle": 1.10,
    "car": 1.50,
    "motorcycle": 1.30,
    "bus": 3.00,
    "truck": 2.80,
    "animal": 0.50,
    "bench": 0.45,
    "chair": 0.90,
    "obstacle on the ground": 0.30,
    "obstacle": 0.80,
    "stop sign": 2.10,
    "traffic light": 2.50,
}

# Mapping YOLO classes -> pedestrian-relevant categories
CATEGORY_MAP = {
    "person": "person",
    "bicycle": "bicycle",
    "car": "car",
    "motorcycle": "motorcycle",
    "bus": "bus",
    "truck": "truck",
    "dog": "animal",
    "cat": "animal",
    "bench": "bench",
    "chair": "chair",
    "backpack": "obstacle on the ground",
    "suitcase": "obstacle on the ground",
    "fire hydrant": "obstacle",
    "stop sign": "stop sign",
    "traffic light": "traffic light",
}

# Danger priority (lower = more urgent)
DANGER_PRIORITY = {
    "car": 1, "bus": 1, "truck": 1, "motorcycle": 1,
    "bicycle": 2, "person": 2, "animal": 2,
    "obstacle": 3, "obstacle on the ground": 3, "bench": 3, "chair": 3,
    "stop sign": 4, "traffic light": 4,
}


def format_distance(distance_m):
    """Formate la distance en mètres ou centimètres."""
    if distance_m is None:
        return "unknown distance"
    if distance_m < 1:
        return f"{int(distance_m * 100)} centimeters"
    return f"{distance_m:.1f} meters"


class ObstacleDetector:
    def __init__(self, model_path="yolov8n.pt"):
        self.model = YOLO(model_path)

    def _estimate_distance_m(self, category, box_height):
        real_height = REAL_HEIGHTS.get(category, 1.0)
        if box_height <= 0:
            return None
        return round((real_height * FOCAL_LENGTH) / box_height, 1)

    def detect(self, frame):
        results = self.model(frame, verbose=False)[0]
        h, w = frame.shape[:2]
        detections = []

        for box in results.boxes:
            cls_id = int(box.cls[0])
            label_en = self.model.names[cls_id]
            if label_en not in CATEGORY_MAP:
                continue

            category = CATEGORY_MAP[label_en]
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2
            box_height = y2 - y1

            if cx < w / 3:
                position = "on your left"
            elif cx > 2 * w / 3:
                position = "on your right"
            else:
                position = "ahead"

            distance_m = self._estimate_distance_m(category, box_height)

            detections.append({
                "category": category,
                "position": position,
                "distance_m": distance_m,
                "confidence": round(conf, 2),
                "priority": DANGER_PRIORITY.get(category, 5),
                "box": (int(x1), int(y1), int(x2), int(y2)),
            })

        detections.sort(key=lambda d: (d["priority"], d["distance_m"] or 999))
        return detections

In [3]:
class SignReader:
    def __init__(self, languages=("en",)):
        self.reader = easyocr.Reader(list(languages), gpu=False)

    def read(self, frame, min_confidence=0.5):
        results = self.reader.readtext(frame)
        texts = []
        for (_, text, conf) in results:
            if conf >= min_confidence and text.strip():
                texts.append({"text": text.strip(), "confidence": round(conf, 2)})
        return texts

In [4]:
class SceneDescriber:
    def __init__(self, model_name="Salesforce/blip-image-captioning-base"):
        self.processor = BlipProcessor.from_pretrained(model_name)
        self.model = BlipForConditionalGeneration.from_pretrained(model_name)

    def describe(self, frame):
        image = Image.fromarray(frame[:, :, ::-1])  # BGR -> RGB
        inputs = self.processor(image, return_tensors="pt")
        out = self.model.generate(**inputs, max_new_tokens=40)
        return self.processor.decode(out[0], skip_special_tokens=True)

In [5]:
class VoiceOutput:
    def __init__(self, rate=175):
        self.rate = rate

    def speak(self, text):
        print(f"[TTS] {text}")
        engine = pyttsx3.init()
        engine.setProperty("rate", self.rate)
        for voice in engine.getProperty("voices"):
            if "en" in voice.id.lower() or "english" in voice.name.lower():
                engine.setProperty("voice", voice.id)
                break
        engine.say(text)
        engine.runAndWait()
        engine.stop()
        del engine

In [6]:
detector = ObstacleDetector()
reader = SignReader()
describer = SceneDescriber()
voice = VoiceOutput()
voice.speak("System ready.")

Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[TTS] System ready.


In [7]:
# Tiens-toi debout à exactement 1 mètre de la caméra, de face, en entier dans le cadre.
KNOWN_DISTANCE_M = 1.0
KNOWN_HEIGHT_M = 1.70   # <-- mets TA taille réelle en mètres

cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    detections_calib = detector.detect(frame)
    persons = [d for d in detections_calib if d["category"] == "person"]
    if persons:
        box_h = persons[0]["box"][3] - persons[0]["box"][1]
        calculated_focal = (box_h * KNOWN_DISTANCE_M) / KNOWN_HEIGHT_M
        print(f"Focale calculée : {calculated_focal:.1f}")
        print("-> Copie cette valeur dans FOCAL_LENGTH (cellule 3), puis redémarre le kernel.")
    else:
        print("Aucune personne détectée, replace-toi bien dans le cadre et relance.")
else:
    print("Erreur caméra")

Focale calculée : 231.8
-> Copie cette valeur dans FOCAL_LENGTH (cellule 3), puis redémarre le kernel.


In [8]:
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    obstacles = detector.detect(frame)
    print("Obstacles:", obstacles)

    texts = reader.read(frame)
    print("Text detected:", texts)

    caption = describer.describe(frame)
    print("Description:", caption)
else:
    print("Camera error")

Obstacles: [{'category': 'person', 'position': 'ahead', 'distance_m': 3.1, 'confidence': 0.81, 'priority': 2, 'box': (101, 94, 514, 479)}]
Text detected: []
Description: a woman with glasses on taking a self


In [9]:
OCR_INTERVAL_SEC = 5
DESCRIBE_INTERVAL_SEC = 15


def format_obstacle_message(detections):
    if not detections:
        return None
    top = detections[:2]
    parts = [f"{d['category']} {format_distance(d['distance_m'])}, {d['position']}" for d in top]
    return "Warning: " + ", ".join(parts)


def draw_detections(frame, detections):
    for d in detections:
        x1, y1, x2, y2 = d["box"]
        dist = d["distance_m"] or 99
        color = (0, 0, 255) if dist < 1.5 else (0, 165, 255) if dist < 4 else (0, 255, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        label = f"{d['category']} - {format_distance(d['distance_m'])} ({d['position']})"
        cv2.putText(frame, label, (x1, max(y1 - 10, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return frame


cap = cv2.VideoCapture(0)
last_ocr = 0
last_describe = 0
last_spoken_msg = None

window_name = "Assist Vision - press 'q' to quit"

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Camera error")
            break

        now = time.time()

        # 1. Obstacle detection
        detections = detector.detect(frame)
        msg = format_obstacle_message(detections)
        if msg and msg != last_spoken_msg:
            voice.speak(msg)
            last_spoken_msg = msg
        elif not msg:
            last_spoken_msg = None

        display_frame = draw_detections(frame.copy(), detections)

        # 2. Sign reading (periodic)
        if now - last_ocr > OCR_INTERVAL_SEC:
            texts = reader.read(frame)
            if texts:
                voice.speak("Sign says: " + texts[0]["text"])
            last_ocr = now

        # 3. Scene description (periodic)
        if now - last_describe > DESCRIBE_INTERVAL_SEC:
            caption = describer.describe(frame)
            voice.speak(caption)
            last_describe = now

        cv2.imshow(window_name, display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            print("Closed by user (q key).")
            break
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            print("Window closed by user.")
            break

except KeyboardInterrupt:
    print("Stopped (Ctrl+C).")
finally:
    cap.release()
    cv2.destroyAllWindows()

[TTS] Warning: person 3.2 meters, ahead
[TTS] a woman with glasses on taking a self self
[TTS] Warning: person 3.3 meters, ahead
[TTS] Warning: person 5.1 meters, ahead
[TTS] a woman in a pink dress standing in a room
[TTS] Warning: person 5.0 meters, ahead
[TTS] a hallway with a white wall and a white ceiling
[TTS] Warning: person 8.6 meters, on your left
[TTS] Warning: person 8.0 meters, on your left
[TTS] a hallway with a white wall and a white door
[TTS] Warning: person 3.8 meters, on your left
[TTS] Warning: person 2.5 meters, ahead
[TTS] Warning: person 2.5 meters, ahead, person 11.5 meters, ahead
[TTS] a person holding a large red ball with white paint
[TTS] a hallway with a white wall and a white ceiling
[TTS] Warning: person 3.8 meters, on your left
[TTS] Warning: person 2.5 meters, on your left
[TTS] Warning: person 2.6 meters, on your left
[TTS] a woman wearing glasses and a pink top
Stopped (Ctrl+C).
